# Laboratorio Evaluado: WBCD y Evaluación Multimodelo

**Reto Autónomo: Clínica Oncológica**

Eres el Consultor IA de un hospital. El dataset WBCD contiene radios y perímetros de células. Debes entrenar k-NN y Árboles de Decisión, y validar qué pasa si no escalas los datos.

## 1. Carga de Datos y Exploración

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

# Carga el dataset WBCD desde UCI (no trae encabezados)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
df_wbcd = pd.read_csv(url, header=None)

# Nombres de columnas: id, diagnosis y 30 características
# (10 medidas x 3 versiones: media, error estándar y peor valor)
medidas = ["radius", "texture", "perimeter", "area", "smoothness",
           "compactness", "concavity", "concave_points", "symmetry",
           "fractal_dimension"]
columnas = ["id", "diagnosis"]
columnas += [m + "_mean" for m in medidas]
columnas += [m + "_se" for m in medidas]
columnas += [m + "_worst" for m in medidas]
df_wbcd.columns = columnas

# Exploración rápida
print("Dimensiones:", df_wbcd.shape)
print(df_wbcd.head())
print("\nConteo de diagnósticos (M = maligno, B = benigno):")
print(df_wbcd["diagnosis"].value_counts())
print("\nValores nulos:", df_wbcd.isnull().sum().sum())

Dimensiones: (569, 32)
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave_points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  radius_worst  texture_worst  pe

## 2. Preprocesamiento

In [2]:
# Convierte la columna de diagnosis (M/B) a binario (1/0)
df_wbcd["diagnosis"] = df_wbcd["diagnosis"].map({"M": 1, "B": 0})

# Separa en X (features) e y (target); el id no sirve como característica
X = df_wbcd.drop(columns=["id", "diagnosis"])
y = df_wbcd["diagnosis"]

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (455, 30) | Test: (114, 30)


## 3. Escalamiento (Solo para k-NN)

In [3]:
# fit_transform() en train, y solo transform() en test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Modelado y Matrices de Confusión

In [4]:
# k-NN con datos escalados
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

# Árbol de Decisión (sin escalar)
arbol = DecisionTreeClassifier(max_depth=4, random_state=42)
arbol.fit(X_train, y_train)
y_pred_arbol = arbol.predict(X_test)

# Matrices de confusión: filas = real, columnas = predicho
# [[VN, FP],
#  [FN, VP]]
print("Matriz de confusión k-NN (escalado):")
print(confusion_matrix(y_test, y_pred_knn))

print("\nMatriz de confusión Árbol de Decisión (sin escalar):")
print(confusion_matrix(y_test, y_pred_arbol))

# Extra: ¿qué pasa si k-NN NO se escala?
knn_sin_escalar = KNeighborsClassifier(n_neighbors=5)
knn_sin_escalar.fit(X_train, y_train)
y_pred_knn_sin = knn_sin_escalar.predict(X_test)

print("\nMatriz de confusión k-NN (SIN escalar):")
print(confusion_matrix(y_test, y_pred_knn_sin))

Matriz de confusión k-NN (escalado):
[[71  1]
 [ 4 38]]

Matriz de confusión Árbol de Decisión (sin escalar):
[[70  2]
 [ 8 34]]

Matriz de confusión k-NN (SIN escalar):
[[71  1]
 [ 9 33]]


## Evaluación Teórica / Decisión de Negocios

1. En este escenario médico, ¿qué implica un Error Tipo II (Falso Negativo) y por qué cuesta vidas?
2. ¿Por qué crees que el Árbol de Decisión no falló estrepitosamente incluso si no hubieses escalado los datos?



**1. Error Tipo II (Falso Negativo)**

Un falso negativo es cuando el modelo dice que el tumor es benigno (0) pero en
realidad es maligno (1). Es decir, el modelo "deja pasar" a una paciente que sí
tiene cáncer. Cuesta vidas porque a esa paciente se le dice que está sana, no
recibe tratamiento a tiempo y la enfermedad avanza hasta una etapa donde es más
difícil o imposible de tratar. En cambio, un falso positivo (Error Tipo I) solo
genera más estudios y un susto, que se corrige con una biopsia. Por eso en
medicina se prefiere un modelo con pocos falsos negativos, aunque tenga algunos
falsos positivos más. Es decir, en este caso importa más el recall (sensibilidad)
que la exactitud global.

**2. Árbol de Decisión sin escalar**

El árbol no depende de distancias. En cada nodo solo se pregunta si una
característica es mayor o menor que un umbral (por ejemplo, "radio > 15"), y se
escoge el umbral que mejor separa las clases. Esa comparación se hace una
característica a la vez, así que la escala (si una variable va de 0 a 0.1 y
otra de 0 a 2500) no cambia cuál es el mejor corte, solo cambia el número del
umbral. En cambio, k-NN calcula distancias entre puntos, entonces una variable
con valores grandes como el área domina la distancia y las demás casi no
cuentan, por eso k-NN sí necesita escalamiento.